<a href="https://colab.research.google.com/github/sreeranjini006/ai-Carrer-guide/blob/main/ibm%20project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>ML Voice Shopping Assistant</title>
<!-- TensorFlow.js -->
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@4.6.0/dist/tf.min.js"></script>
<style>
    body {font-family: 'Poppins', sans-serif; margin:0; background:#f5f5f5;}
    header {background:#4CAF50; color:white; padding:15px; text-align:center; font-size:24px; font-weight:600;}
    #voiceBtn {margin:20px auto; display:block; padding:10px 20px; font-size:18px; background:#4CAF50; color:white; border:none; border-radius:5px; cursor:pointer;}
    #output {text-align:center; margin:10px 0; font-weight:600; color:#333;}
    #products {display:flex; flex-wrap:wrap; justify-content:center; gap:15px; padding:10px;}
    .card {background:white; width:180px; border-radius:10px; padding:10px; text-align:center; box-shadow:0 2px 6px rgba(0,0,0,0.2);}
    .card img {width:100%; border-radius:10px;}
    .card button {margin-top:10px; padding:8px; width:100%; border:none; background:#4CAF50; color:white; border-radius:5px; cursor:pointer;}
    #cart {margin:20px auto; width:90%; max-width:600px; background:white; padding:10px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.2);}
    #cart h3 {margin:0 0 10px 0; text-align:center;}
    #cart ul {list-style:none; padding:0; margin:0;}
    #cart ul li {padding:5px 0; border-bottom:1px solid #ddd;}
</style>
</head>
<body>

<header>ML Voice Shopping Assistant</header>

<button id="voiceBtn">🎤 Speak</button>
<p id="output">Say: "Show shoes", "Add first item to cart", or "Checkout"</p>

<div id="products"></div>

<div id="cart">
    <h3>Cart Items</h3>
    <ul id="cartList"></ul>
</div>

<script>
// Sample products
const products = [
    {name:"Nike Shoes", price:1999, img:"https://via.placeholder.com/150?text=Shoes"},
    {name:"Adidas T-Shirt", price:499, img:"https://via.placeholder.com/150?text=T-Shirt"},
    {name:"Samsung Mobile", price:15000, img:"https://via.placeholder.com/150?text=Mobile"},
    {name:"Levi's Jeans", price:1200, img:"https://via.placeholder.com/150?text=Jeans"},
    {name:"Puma Sneakers", price:2500, img:"https://via.placeholder.com/150?text=Sneakers"}
];

let cart = [];

// Display products
function showProducts(list){const productsDiv=document.getElementById("products");productsDiv.innerHTML="";list.forEach((p,index)=>{let card=document.createElement("div");card.className="card";card.innerHTML=`<img src="${p.img}" alt="${p.name}"><h4>${p.name}</h4><p>₹${p.price}</p><button onclick="addToCart(${index})">Add to Cart</button>`;productsDiv.appendChild(card);});}
showProducts(products);

// Cart functions
function addToCart(index){cart.push(products[index]);updateCart();speak(`${products[index].name} added to cart`);}
function updateCart(){const cartList=document.getElementById("cartList");cartList.innerHTML="";cart.forEach(item=>{let li=document.createElement("li");li.textContent=`${item.name} - ₹${item.price}`;cartList.appendChild(li);});}

// ML-based intent recognition (TensorFlow.js)
// Simple bag-of-words classifier for demo
const intents = [
    {intent:"search", examples:["show shoes","show t-shirt","show mobile","show sneakers"]},
    {intent:"add_cart", examples:["add first item","add to cart","add sneakers"]},
    {intent:"remove_cart", examples:["remove first item","remove t-shirt"]},
    {intent:"checkout", examples:["checkout","proceed to checkout","buy now"]}
];

// Build vocabulary
let vocab = [];
intents.forEach(i=>i.examples.forEach(e=>{e.split(" ").forEach(w=>{if(!vocab.includes(w)) vocab.push(w);});}));

// Convert text to vector
function textToVector(text){text=text.toLowerCase();let vec=new Array(vocab.length).fill(0);text.split(" ").forEach(w=>{let idx=vocab.indexOf(w);if(idx>=0) vec[idx]=1;});return vec;}

// Prepare training data
let xs=[], ys=[];
intents.forEach((i,index)=>{i.examples.forEach(ex=>{xs.push(textToVector(ex));ys.push(index);});});
const xsTensor = tf.tensor2d(xs);
const ysTensor = tf.oneHot(tf.tensor1d(ys,'int32'),intents.length);

// Build model
const model = tf.sequential();
model.add(tf.layers.dense({units:16,inputShape:[vocab.length],activation:'relu'}));
model.add(tf.layers.dense({units:intents.length,activation:'softmax'}));
model.compile({loss:'categoricalCrossentropy',optimizer:'adam',metrics:['accuracy']});

// Train model
async function trainModel(){await model.fit(xsTensor,ysTensor,{epochs:200});}
trainModel();

// Predict intent
async function predictIntent(text){
    const input = tf.tensor2d([textToVector(text)]);
    const prediction = model.predict(input);
    const index = prediction.argMax(1).dataSync()[0];
    return intents[index].intent;
}

// Voice recognition
document.getElementById("voiceBtn").addEventListener("click",()=>{
    const recognition=new (window.SpeechRecognition || window.webkitSpeechRecognition)();
    recognition.start();
    recognition.onresult=async function(event){
        const text=event.results[0][0].transcript.toLowerCase();
        document.getElementById("output").innerText=`You said: "${text}"`;
        const intent=await predictIntent(text);
        handleIntent(intent, text);
    }
});

// Handle predicted intent
function handleIntent(intent, text){
    if(intent==="search"){
        let filtered = products.filter(p=>text.includes(p.name.toLowerCase().split(" ")[0]));
        if(filtered.length===0) filtered=products;
        showProducts(filtered);
        speak("Showing products");
    } else if(intent==="add_cart"){
        addToCart(0);
    } else if(intent==="remove_cart"){
        if(cart.length>0){cart.pop();updateCart();speak("Removed last item from cart");}else speak("Cart is empty");
    } else if(intent==="checkout"){
        if(cart.length>0) speak(`You have ${cart.length} items. Proceeding to checkout`); else speak("Cart is empty");
    } else speak("Sorry, I did not understand");
}

// Text-to-speech
function speak(text){let speech=new SpeechSynthesisUtterance(text);speechSynthesis.speak(speech);}
</script>

</body>
</html>
"""

HTML(html_content)

In [6]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>ML Voice Shopping Assistant</title>

<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@4.6.0/dist/tf.min.js"></script>

<style>
    body {
        font-family: 'Poppins', sans-serif;
        margin:0;
        background:#f5f5f5;
        color:#2c3e50; /* Changed overall text color */
    }

    header {
        background:#4CAF50;
        color:white;
        padding:15px;
        text-align:center;
        font-size:24px;
        font-weight:600;
    }

    #voiceBtn {
        margin:20px auto;
        display:block;
        padding:10px 20px;
        font-size:18px;
        background:#4CAF50;
        color:white;
        border:none;
        border-radius:5px;
        cursor:pointer;
    }

    #output {
        text-align:center;
        margin:10px 0;
        font-weight:600;
        color:#2c3e50; /* Changed from grey to dark blue */
    }

    #products {
        display:flex;
        flex-wrap:wrap;
        justify-content:center;
        gap:15px;
        padding:10px;
    }

    .card {
        background:white;
        width:180px;
        border-radius:10px;
        padding:10px;
        text-align:center;
        box-shadow:0 2px 6px rgba(0,0,0,0.2);
    }

    .card img {
        width:100%;
        border-radius:10px;
    }

    .card button {
        margin-top:10px;
        padding:8px;
        width:100%;
        border:none;
        background:#4CAF50;
        color:white;
        border-radius:5px;
        cursor:pointer;
    }

    #cart {
        margin:20px auto;
        width:90%;
        max-width:600px;
        background:white;
        padding:10px;
        border-radius:10px;
        box-shadow:0 2px 6px rgba(0,0,0,0.2);
    }

    #cart h3 {
        margin:0 0 10px 0;
        text-align:center;
    }

    #cart ul {
        list-style:none;
        padding:0;
        margin:0;
    }

    #cart ul li {
        padding:5px 0;
        border-bottom:1px solid #ddd;
    }
</style>
</head>

<body>

<header>ML Voice Shopping Assistant</header>

<button id="voiceBtn">🎤 Speak</button>
<p id="output">Say: "Show shoes", "Add first item to cart", or "Checkout"</p>

<div id="products"></div>

<div id="cart">
    <h3>Cart Items</h3>
    <ul id="cartList"></ul>
</div>

<script>
const products = [
    {name:"Nike Shoes", price:1999, img:"https://via.placeholder.com/150?text=Shoes"},
    {name:"Adidas T-Shirt", price:499, img:"https://via.placeholder.com/150?text=T-Shirt"},
    {name:"Samsung Mobile", price:15000, img:"https://via.placeholder.com/150?text=Mobile"},
    {name:"Levi's Jeans", price:1200, img:"https://via.placeholder.com/150?text=Jeans"},
    {name:"Puma Sneakers", price:2500, img:"https://via.placeholder.com/150?text=Sneakers"}
];

let cart = [];

function showProducts(list){
    const productsDiv=document.getElementById("products");
    productsDiv.innerHTML="";
    list.forEach((p,index)=>{
        let card=document.createElement("div");
        card.className="card";
        card.innerHTML=`
            <img src="${p.img}">
            <h4>${p.name}</h4>
            <p>₹${p.price}</p>
            <button onclick="addToCart(${index})">Add to Cart</button>
        `;
        productsDiv.appendChild(card);
    });
}
showProducts(products);

function addToCart(index){
    cart.push(products[index]);
    updateCart();
    speak(products[index].name + " added to cart");
}

function updateCart(){
    const cartList=document.getElementById("cartList");
    cartList.innerHTML="";
    cart.forEach(item=>{
        let li=document.createElement("li");
        li.textContent=item.name + " - ₹" + item.price;
        cartList.appendChild(li);
    });
}

// Voice recognition
document.getElementById("voiceBtn").addEventListener("click",()=>{
    const recognition=new (window.SpeechRecognition || window.webkitSpeechRecognition)();
    recognition.start();

    recognition.onresult=async function(event){
        const text=event.results[0][0].transcript.toLowerCase();
        document.getElementById("output").innerText="You said: " + text;
        processCommand(text);
    }
});

function processCommand(command){
    if(command.includes("show")){
        let keyword=command.replace("show","").trim();
        let filtered=products.filter(p=>p.name.toLowerCase().includes(keyword));
        showProducts(filtered.length ? filtered : products);
        speak("Showing " + keyword);
    }
    else if(command.includes("add first item")){
        addToCart(0);
    }
    else if(command.includes("checkout")){
        speak(cart.length ? "Proceeding to checkout" : "Cart is empty");
    }
    else{
        speak("Command not understood");
    }
}

function speak(text){
    let speech=new SpeechSynthesisUtterance(text);
    speechSynthesis.speak(speech);
}
</script>

</body>
</html>
"""

HTML(html_content)